# Edge Cases

Runs dynamic, recurrent, and expected-failure paths that protect against common audit regressions.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.utils.iter_accessible_attributes",
    "tl.utils.list_modules",
    "tl.utils.list_ops",
    "tl.utils.log_current_autocast_state",
    "tl.utils.log_current_rng_states",
    "tl.utils.log_forward_pass_streaming",
    "tl.utils.make_random_barcode",
    "tl.utils.make_short_barcode_from_input",
    "tl.utils.nested_assign",
    "tl.utils.nested_getattr",
    "tl.utils.normalize_input_args",
    "tl.utils.peek_graph",
    "tl.utils.print_override",
    "tl.utils.progress_bar",
    "tl.utils.register_op_rule",
    "tl.utils.remove_attributes_with_prefix",
    "tl.utils.remove_entry_from_list",
    "tl.utils.safe_copy",
    "tl.utils.safe_copy_args",
    "tl.utils.safe_copy_kwargs",
    "tl.utils.safe_to",
    "tl.utils.set_random_seed",
    "tl.utils.set_rng_from_saved_states",
    "tl.utils.synthetic_input",
    "tl.utils.tensor_all_nan",
    "tl.utils.tensor_nanequal",
    "tl.utils.tensor_stats_summary",
    "tl.utils.warn_parallel",
    "tl.validate",
    "tl.validation",
    "tl.validation.InterventionValidationReport",
    "tl.validation.MetadataInvariantError",
    "tl.validation.check_metadata_invariants",
    "tl.validation.check_spec_compat",
    "tl.validation.resolve_sites",
    "tl.validation.validate",
    "tl.validation.validate_backward_pass",
    "tl.validation.validate_batch_of_models_and_inputs",
    "tl.validation.validate_forward_pass",
    "tl.validation.validate_model_log_saved_activations",
    "tl.validation.validate_saved_activations",
    "tl.validation.validate_tlspec",
    "tl.viewer",
    "tl.visualization",
    "tl.visualization.NodeSpec",
    "tl.visualization.render_lines_to_html",
    "tl.visualization.show_backward_graph",
    "tl.visualization.show_bundle_graph",
    "tl.visualization.show_model_graph",
    "tl.visualization.summary",
    "tl.viz",
    "tl.viz.bundle_diff",
    "tl.viz.causal_trace_heatmap",
    "tl.where",
    "tl.zero_ablate",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")